# POS-теггер для русского языка, устойчивый к опечаткам

## 📋 Постановка бизнес-задачи

**Контекст**  
Вы работаете в компании, разрабатывающей интеллектуальную систему проверки грамматики и стиля для русского языка «Грамотей». Пользователи часто вводят текст с опечатками (например, «карова» вместо «корова», «здарова» вместо «здорово»), из‑за чего текущий модуль анализа морфологии выдаёт ошибки. Нужен прототип, который **корректно определяет UPOS-тег для каждого слова даже в предложениях с опечатками**.

**Задача**  
Разработать **POS-теггер для русского языка, устойчивый к опечаткам**.

- **Вход:** последовательность токенов (слова, возможно с опечатками).
- **Выход:** последовательность UPOS-тегов (17 классов UD).

**Метрика:** Token Accuracy на `test_noisy`.  
**Бизнес-критерий:** Accuracy ≥ **0.88**.

**Подход:**
- Кандидаты исправлений: **Trie + BFS** (расстояние Дамерау–Левенштейна ≤ `max_dist`).
- Вероятность опечатки: **ErrorModel** (практика 31).
- Контекст тегов: **HMM + Витерби** (NLTK, практика 32).

> **Важно:** не используйте `pyspellchecker` и другие готовые spell-check библиотеки. DL, Trie, ErrorModel — вручную. HMM — NLTK.


## 📊 Данные

### 1. Universal Dependencies — Russian SynTagRus
```python
from datasets import load_dataset
dataset = load_dataset("commul/universal_dependencies", "ru_syntagrus")
```
~48k train, ~6k dev, ~6k test. Поля: `tokens`, `upos`, `lemmas`, `feats`.

### 2. Kartaslov orfo_and_typos (практика 31)
Локальный файл: `data/orfo_and_typos.csv` (столбцы: `CORRECT`, `MISTAKE`, `WEIGHT`).

### Оценка
- `train`, `dev` — чистые предложения (обучение HMM).
- `test_clean` — оригинальный test.
- `test_noisy` — test с опечатками (p=0.15 на слово; операции: sub 0.5, del 0.2, ins 0.2, trans 0.1). Gold-теги — из чистого test.


---
## Подготовка окружения

Заполните ячейку ниже: импорты, константы, пути к данным.

Рекомендуемые константы: `RANDOM_STATE`, `MIN_WORD_COUNT`, `UNK_TOKEN`, `NUM_TOKEN`, `TARGET_ACCURACY=0.80`, `TYPO_PROB=0.15`, `ALPHA_ERROR=0.01`.


In [2]:
# импорты, константы, создание папки artifacts

import os
from pathlib import Path
import json
import random

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import re
import math
import time

from datasets import load_dataset
from nltk.tag.hmm import HiddenMarkovModelTrainer
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import pickle

from collections import Counter,defaultdict, deque
from typing import Dict, List, Optional, Tuple


RANDOM_STATE = 42
MIN_WORD_COUNT = 2
UNK_TOKEN = '<UNK>'
NUM_TOKEN = '<NUM>'
TARGET_ACCURACY = 0.88
TYPO_PROB = 0.15
GAMMA = 0.1
MAX_EDIT_DIST = 2
ALPHA_ERROR = 0.01    # сглаживание ErrorModel (шаг 3)
ALPHA_LM = 1e-6       # сглаживание языковой модели, Лаплас (шаг 6)
TEST_SAMPLE = 100_000   # размер тестовой подвыборки для оценки
DATA_DIR = Path("data")
ARTIFACTS_DIR = Path("artifacts")
TYPOS_PATH = DATA_DIR/'orfo_and_typos.csv'
DATA_DIR.mkdir(exist_ok=True)
ARTIFACTS_DIR.mkdir(exist_ok=True)

np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

print("Готово. Целевая accuracy:", TARGET_ACCURACY)

C:\Users\admin\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Готово. Целевая accuracy: 0.88


---
## Шаг 1. Загрузка данных и первичный осмотр

- Загрузите `ru_syntagrus` через `datasets`.
- Выведите число предложений и токенов в каждом сплите.
- Загрузите CSV опечаток, покажите `head()`, проверьте пропуски.


In [3]:
# загрузка UD (ru_syntagrus)
dataset_ru_syntagrus = load_dataset("commul/universal_dependencies", "ru_syntagrus")

In [27]:
print(dataset_ru_syntagrus)
print('-'*100)
print(f'Dataset Keys: {dataset_ru_syntagrus.keys()}')
print('-'*100)
print(f'Train Len = {len(dataset_ru_syntagrus["train"])} , Test Len = {len(dataset_ru_syntagrus["test"])}, Val Len = {len(dataset_ru_syntagrus["dev"])}')
print('-'*100)
print(f'{dataset_ru_syntagrus['train'][1]}')
print('-'*100)
print(f'{sum(len(t) for t in dataset_ru_syntagrus["train"]["tokens"])} - Train Tokens, {sum(len(t) for t in dataset_ru_syntagrus["test"]["tokens"])} - Test Tokens, '
      f'{sum(len(t) for t in dataset_ru_syntagrus["dev"]["tokens"])} - Validation Tokens')
print('-'*100)
dataset_ru_syntagrus["train"].features['upos']

DatasetDict({
    dev: Dataset({
        features: ['sent_id', 'text', 'comments', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'head', 'deprel', 'deps', 'misc', 'mwt', 'empty_nodes'],
        num_rows: 8906
    })
    test: Dataset({
        features: ['sent_id', 'text', 'comments', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'head', 'deprel', 'deps', 'misc', 'mwt', 'empty_nodes'],
        num_rows: 8800
    })
    train: Dataset({
        features: ['sent_id', 'text', 'comments', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'head', 'deprel', 'deps', 'misc', 'mwt', 'empty_nodes'],
        num_rows: 69631
    })
})
----------------------------------------------------------------------------------------------------
Dataset Keys: dict_keys(['dev', 'test', 'train'])
----------------------------------------------------------------------------------------------------
Train Len = 69631 , Test Len = 8800, Val Len = 8906
------------------------------------------------------------------------

List(Value('string'))

In [13]:
tags_lst = sorted({tag for sent in dataset_ru_syntagrus["train"]["upos"] for tag in sent})
print(tags_lst)

['ADJ', 'ADP', 'ADV', 'AUX', 'CCONJ', 'DET', 'INTJ', 'NOUN', 'NUM', 'PART', 'PRON', 'PROPN', 'PUNCT', 'SCONJ', 'SYM', 'VERB', 'X']


In [15]:
# загрузка orfo_and_typos.csv
import urllib.request

TYPOS_URL = ("https://raw.githubusercontent.com/dkulagin/kartaslov/"
             "master/dataset/orfo_and_typos/orfo_and_typos.L1_5.csv")

if not TYPOS_PATH.exists():
    print("Скачиваю датасет опечаток...")
    urllib.request.urlretrieve(TYPOS_URL, TYPOS_PATH)
    print("Готово")
else:
    print("Файл уже на месте")

Скачиваю датасет опечаток...
Готово


In [23]:
orfo_df = pd.read_csv(TYPOS_PATH, sep=";")
print(orfo_df.head())
print('-'*100)
print(orfo_df.describe())

  CORRECT  MISTAKE  WEIGHT
0  болото   болотл  0.3333
1  болото  болотао  0.2500
2  болото   балото  0.1219
3  болото    болто  0.0562
4  болото  болотаъ  0.0526
----------------------------------------------------------------------------------------------------
             WEIGHT
count  85550.000000
mean       0.198534
std        0.170689
min        0.000000
25%        0.052600
50%        0.142900
75%        0.333300
max        0.947400


In [48]:
orfo_df.columns = ["correct", "typo", "weight"]
print("Колонки:", orfo_df.columns.tolist())
print('-'*100)
print("Пропуски:\n", orfo_df.isnull().sum())
print('-'*100)
print("Дубликаты:", orfo_df.duplicated().sum())
print('-'*100)
print("Нулевые значения:", sum(orfo_df['weight']==0))
print('-'*100)
print("Совпадения correct и typo:", sum(orfo_df['typo']==orfo_df['correct']))
print('-'*100)
print("Посторонние символы в correct:", orfo_df["correct"].str.contains(r"[^а-яё]").sum())
print('-'*100)
print("Посторонние символы в typo:", orfo_df["typo"].str.contains(r"[^а-яё]").sum())
print('-'*100)
print(orfo_df[orfo_df["correct"].str.contains(r"[^а-яё]")].head())
print('-'*100)
print((orfo_df["correct"] != orfo_df["correct"].str.lower()).sum())
orfo_df.describe(include="all")

Колонки: ['correct', 'typo', 'weight']
----------------------------------------------------------------------------------------------------
Пропуски:
 correct    0
typo       0
weight     0
dtype: int64
----------------------------------------------------------------------------------------------------
Дубликаты: 0
----------------------------------------------------------------------------------------------------
Нулевые значения: 1
----------------------------------------------------------------------------------------------------
Совпадения correct и typo: 0
----------------------------------------------------------------------------------------------------
Посторонние символы в correct: 72
----------------------------------------------------------------------------------------------------
Посторонние символы в typo: 103
----------------------------------------------------------------------------------------------------
               correct             typo  weight
11924          

,correct,typo,weight
count,85550,85550,85550.000000
unique,21749,69338,NaN
top,что,см,NaN
freq,122,51,NaN
mean,NaN,NaN,0.198534
std,NaN,NaN,0.170689
min,NaN,NaN,0.000000
25%,NaN,NaN,0.052600
50%,NaN,NaN,0.142900
75%,NaN,NaN,0.333300


---
## Шаг 2. EDA

- Гистограмма длин предложений (train).
- Barplot частот UPOS-тегов.
- Топ-10 омонимов (слова с >1 тегом).
- Гистограмма DL между correct и typo.
- (Опционально) частоты операций sub/ins/del/trans в ErrorModel.


In [ ]:
# TODO: EDA — длины предложений и частоты тегов



In [ ]:
# TODO: EDA — омонимы



In [ ]:
# TODO: EDA — опечатки (DL, типы операций)



---
## Шаг 3. Предобработка

- lower case для токенов UD.
- Частотный словарь по train; редкие слова (<2) → `<UNK>`, цифры → `<NUM>`.
- Очистка опечаток: только кириллица, без дубликатов, correct ≠ typo.
- Обучите `ErrorModel` (α=0.01) на парах (correct, typo, weight).


In [ ]:
# TODO: предобработка токенов UD, формат [(слово, тег), ...]



In [ ]:
# TODO: очистка датасета опечаток + обучение ErrorModel



---
## Шаг 4. Зашумлённый тест

- `train_sents`, `dev_sents`, `test_clean_sents` — чистые.
- Сгенерируйте `test_noisy` (p=0.15) с помощью ErrorModel.
- Сохраните gold-теги для оценки.


In [ ]:
# TODO: функция generate_typo + создание test_noisy



---
## Шаг 5. Trie и кандидаты

- Словарь из train + `<UNK>`, `<NUM>`.
- Классы `Trie`, `TrieNode` и функция `trie_candidates(word, max_dist=2)` (BFS).
- Проверьте на примерах: «здарова», «привте».


In [ ]:
# TODO: Trie + trie_candidates (BFS, DL)



In [ ]:
# TODO: демо кандидатов на нескольких опечатках



---
## Шаг 6. Языковая модель P(w)

- Частоты слов по train.
- Сглаживание Лапласа (α=1e-6) → `log P(word)`.


In [ ]:
# TODO: log_word_prior для всех слов словаря



---
## Шаг 7. HMM на чистых предложениях (NLTK)

- `HiddenMarkovModelTrainer` + `LidstoneEstimator(gamma=0.1)` (см. `lidstone_estimator.py`).
- Формат обучения: `List[List[Tuple[str, str]]]`.
- Accuracy на чистом `dev`.


In [ ]:
# TODO: обучение HMM (NLTK) + оценка на dev (clean)



---
## Шаг 8. Baseline — независимое исправление + HMM

Для каждого слова в `test_noisy`:
1. Найдите лучший кандидат (min DL, tie-break по частоте).
2. Примените `tagger.tag()` к исправленному предложению.
3. Посчитайте Token Accuracy.


In [ ]:
# TODO: correct_independent + baseline accuracy на test_noisy



---
## Шаг 9. Гибридный метод — расширенный Витерби

Для каждого наблюдаемого слова:
- Кандидаты из `trie_candidates` (+ `<UNK>` если пусто).
- `score = log P(опечатка|кандидат) + log P(кандидат|тег)` (эмиссия из HMM).

Модифицируйте Витерби: на каждом шаге перебирайте только кандидаты.  
Верните последовательность тегов и исправленных слов. Оцените accuracy.


In [ ]:
# TODO: build_candidate_list + viterbi_candidates



In [ ]:
# TODO: гибридный метод — accuracy на test_noisy + сравнение с baseline



---
## Шаг 10. Настройка гиперпараметров

- Grid search `gamma` ∈ {0.01, 0.05, 0.1, 0.5, 1.0, 2.0} на чистом dev.
- Grid search `max_dist` ∈ {1, 2} на `test_noisy`.
- (Опционально) порог `log P(опечатка|кандидат) < -5`.


In [ ]:
# TODO: grid search gamma (dev clean)



In [ ]:
# TODO: grid search max_dist (test_noisy)



---
## Шаг 11. Финальная оценка

- Обучите HMM на `train + dev` с лучшим `gamma`.
- Гибридный метод на `test_noisy`.
- Метрики: accuracy, F1-micro, F1-macro, classification_report.
- Confusion matrix.
- 10 примеров: noisy → corrected → gold tags → pred tags.


In [ ]:
# TODO: финальная модель + метрики на test_noisy



In [ ]:
# TODO: confusion matrix



In [ ]:
# TODO: 10 примеров предложений с опечатками



---
## Шаг 12. Анализ ошибок и выводы

- Доля слов без кандидатов (DL > max_dist).
- Ошибки на омонимах и редких словах.
- Случаи: неверное исправление / верный тег (и наоборот).
- Латентность: baseline vs гибрид.
- Заключение: достигнут ли критерий 0.88? Что улучшить?


In [ ]:
# TODO: доля слов без кандидатов



In [ ]:
# TODO: сравнение латентности baseline vs гибрид



In [ ]:
# TODO: сохранение артефактов (eval_metrics.json, hmm_tagger.pkl)



## Выводы

_Заполните после выполнения всех шагов:_

1. Достигнут ли бизнес-критерий (Accuracy ≥ 0.88 на `test_noisy`)?
2. Какой метод лучше — baseline или гибридный Витерби? Почему?
3. Как опечатки влияют на accuracy по сравнению с `test_clean`?
4. Какие улучшения предложите (клавиатурная раскладка, символьные признаки, нейросети)?
